# Matching - PySpark version

In [1]:
# matching.py

import os
import json
import numpy as np
import pandas as pd

from typing import List, Optional, Dict, Any

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

StatementMeta(, 0a5580ee-215b-42a5-95c9-a19456250d9a, 3, Finished, Available, Finished, False)

In [2]:
%run ./matching

StatementMeta(, 0a5580ee-215b-42a5-95c9-a19456250d9a, 4, Finished, Available, Finished, True)

In [5]:
month_result= pd.read_parquet("/lakehouse/default/Files/month_data")
print(month_result.shape)
month_result_spark= spark.createDataFrame(month_result)

StatementMeta(, 0a5580ee-215b-42a5-95c9-a19456250d9a, 7, Finished, Available, Finished, False)

(4538578, 17)


In [ ]:
print("Running time series matching...")

res_ts = run_time_series_matching_pipeline(
    sdf=month_result_spark,
    output_folder="Files/output/matching/time_series",
    lookback_months=12,
    verbose=True,
    save_output=False
)

# res_ts["balance"].show(5)

StatementMeta(, 0a5580ee-215b-42a5-95c9-a19456250d9a, 9, Finished, Available, Finished, False)

Running time series matching...
Preparing base dataframe ...
Building risk set rows ...
risk_rows count = 5736364
+----------+-------+------+
|Ti        |group  |count |
+----------+-------+------+
|2025-03-01|control|578944|
|2025-03-01|treated|7495  |
|2025-04-01|control|575007|
|2025-04-01|treated|4008  |
|2025-05-01|control|573401|
|2025-05-01|treated|1680  |
|2025-06-01|control|571970|
|2025-06-01|treated|1500  |
|2025-07-01|control|571260|
|2025-07-01|treated|768   |
|2025-08-01|control|570833|
|2025-08-01|treated|480   |
|2025-09-01|control|570295|
|2025-09-01|treated|588   |
|2025-10-01|control|569738|
|2025-10-01|treated|600   |
|2025-11-01|control|567999|
|2025-11-01|treated|1773  |
|2025-12-01|control|566573|
|2025-12-01|treated|1452  |
+----------+-------+------+

Building fixed-lag time series profiles ...
profiles count = 477628
Standardizing by controls ...
profiles_z count = 477628
Matching top-k ...
matches count = 8470
Building matched profiles ...
matched_profiles co

In [ ]:
save_matching_results_fabric(
    res=res_ts,
    folder="Files/output/matching/time_series",
    config={
        "type": "time_series",
        "lookback_months": 12
    },
)

love_plot_from_spark(
    res_ts["balance"],
    output_path=None,   # 👈 不存檔
    title="Time Series Matching"
)

StatementMeta(, 0a5580ee-215b-42a5-95c9-a19456250d9a, 10, Finished, Available, Finished, False)

Saving parquet to Files/output/matching/time_series ...
✅ Save completed


In [ ]:
print("Running summary matching 1...")

res_summary_1 = run_summary_matching_pipeline(
    sdf=month_result,
    output_folder="Files/output/matching/summary_1",
    lookback_months=12,
    summary_vars=[
        "peak_mean",
        "mean_consumption",
        "peak_sd",
        "trend"
    ],
    verbose=True,
    save_output=False
)

res_summary_1["balance"].show(5)

save_matching_results_fabric(
    res=res_summary_1,
    folder="Files/output/matching/summary_1",
    config={
        "type": "summary",
        "vars": ["peak_mean","mean_consumption","peak_sd","trend"]
    },
    plot_title="Summary Matching 1"
)

In [ ]:
print("Running summary matching 2...")

res_summary_2 = run_summary_matching_pipeline(
    sdf=month_result,
    output_folder="Files/output/matching/summary_2",
    lookback_months=12,
    summary_vars=[
        "peak_mean",
        "mean_consumption",
        "variance_consumption",
        "trend"
    ],
    verbose=True,
    save_output=False
)

res_summary_2["balance"].show(5)

save_matching_results_fabric(
    res=res_summary_2,
    folder="Files/output/matching/summary_2",
    config={
        "type": "summary",
        "vars": ["peak_mean","mean_consumption","variance_consumption","trend"]
    },
    plot_title="Summary Matching 2"
)

In [ ]:
print("Running summary matching 3...")

res_summary_3 = run_summary_matching_pipeline(
    sdf=month_result,
    output_folder="Files/output/matching/summary_3",
    lookback_months=12,
    summary_vars=[
        "peak_mean",
        "peak_sd",
        "trend"
    ],
    verbose=True,
    save_output=False
)

res_summary_3["balance"].show(5)

save_matching_results_fabric(
    res=res_summary_3,
    folder="Files/output/matching/summary_3",
    config={
        "type": "summary",
        "vars": ["peak_mean","peak_sd","trend"]
    },
    plot_title="Summary Matching 3"
)

In [ ]:
print("Running seasonal summary matching...")

res_summary_season = run_summary_matching_pipeline(
    sdf=month_result,
    output_folder="Files/output/matching/summary_season",
    lookback_months=24,
    match_months=[1,2,3,11,12],
    summary_vars=[
        "peak_mean",
        "peak_sd",
        "peak_volatility",
        "trend"
    ],
    verbose=True,
    save_output=False
)

res_summary_season["balance"].show(5)

save_matching_results_fabric(
    res=res_summary_season,
    folder="Files/output/matching/summary_season",
    config={
        "type": "summary_season",
        "match_months": [1,2,3,11,12]
    },
    plot_title="Summary Matching (High Price Period)"
)

In [ ]:
print("Running calendar matching...")

res_calendar = run_time_series_matching_pipeline(
    sdf=month_result,
    output_folder="Files/output/matching/calendar",
    lookback_months=24,
    match_months=[1,2,3,11,12],
    verbose=True,
    save_output=False
)

res_calendar["balance"].show(5)

save_matching_results_fabric(
    res=res_calendar,
    folder="Files/output/matching/calendar",
    config={
        "type": "calendar",
        "match_months": [1,2,3,11,12]
    },
    plot_title="Calendar Matching"
)

## Load Matching result

In [ ]:
matches = spark.read.parquet("Files/output/matching/time_series/matches")
display(matches)

profiles = spark.read.parquet("Files/output/matching/time_series/profiles")
display(profiles)